# 분개장과 마스터 데이터 대조

분개장에 입력된 계정과목, 거래처 및 부서 코드가
각 기준정보에 존재하는지 확인한다.

## 1. 기준정보와 분개장 불러오기

In [1]:
# 데이터 처리와 경로 설정에 필요한 라이브러리
import pandas as pd
from pathlib import Path


# 현재 실행 위치를 기준으로 프로젝트 루트 설정
current_path = Path.cwd()

if current_path.name == "notebooks":
    project_root = current_path.parent
else:
    project_root = current_path

# 각 파일의 경로 설정
accounts_path = project_root / "data" / "master" / "accounts.csv"
vendors_path = project_root / "data" / "master" / "vendors.csv"
departments_path = project_root / "data" / "master" / "departments.csv"
journal_path = project_root / "data" / "raw" / "journal_sample.xlsx"

# 계정과목, 거래처, 부서 기준표 불러오기
accounts = pd.read_csv(accounts_path)
vendors = pd.read_csv(vendors_path)
departments = pd.read_csv(departments_path)

# 정상 분개장 예시 불러오기
journal = pd.read_excel(
    journal_path,
    sheet_name="분개장"
)

print("계정과목 수:", len(accounts))
print("거래처 수:", len(vendors))
print("부서 수:", len(departments))
print("분개장 거래 수:", len(journal))

계정과목 수: 22
거래처 수: 12
부서 수: 7
분개장 거래 수: 10


## 2. 유효한 기준 코드 집합 생성

In [2]:
# 계정과목 코드는 숫자 형태로 통일한 후 유효 코드 집합 생성
valid_account_codes = set(
    pd.to_numeric(accounts["account_code"], errors="coerce")
    .dropna()
    .astype(int)
)

# 거래처 코드와 부서 코드의 유효 집합 생성
valid_partner_codes = set(
    vendors["partner_code"].dropna().astype(str)
)

valid_department_codes = set(
    departments["department_code"].dropna().astype(str)
)

print("계정과목 코드 예시:", sorted(valid_account_codes)[:5])
print("거래처 코드 예시:", sorted(valid_partner_codes)[:5])
print("부서 코드 예시:", sorted(valid_department_codes)[:5])

계정과목 코드 예시: [1100, 1110, 1130, 1180, 1210]
거래처 코드 예시: ['C001', 'C002', 'C003', 'C004', 'V001']
부서 코드 예시: ['D001', 'D002', 'D003', 'D004', 'D005']


## 3. 분개장 코드 유효성 검사

In [3]:
# 분개장에서 계정코드가 들어 있는 열 목록
account_columns = [
    "debit_account_1",
    "debit_account_2",
    "credit_account_1",
    "credit_account_2"
]

# 발견된 오류를 저장할 빈 목록
validation_errors = []

# 각 계정코드 열을 순서대로 검사
for column in account_columns:
    for row_index, value in journal[column].items():

        # 두 번째 계정이 필요하지 않은 거래의 빈칸은 오류가 아님
        if pd.isna(value):
            continue

        # 엑셀에서 숫자가 실수로 읽힐 수 있으므로 정수로 변환
        account_code = int(value)

        # 기준표에 존재하지 않는 계정코드 기록
        if account_code not in valid_account_codes:
            validation_errors.append({
                "row_number": row_index + 2,
                "voucher_id": journal.loc[row_index, "voucher_id"],
                "column": column,
                "invalid_value": account_code,
                "error_type": "존재하지 않는 계정과목"
            })

# 거래처 코드 검사
for row_index, value in journal["partner_code"].items():
    if pd.isna(value) or str(value) not in valid_partner_codes:
        validation_errors.append({
            "row_number": row_index + 2,
            "voucher_id": journal.loc[row_index, "voucher_id"],
            "column": "partner_code",
            "invalid_value": value,
            "error_type": "존재하지 않는 거래처"
        })

# 부서 코드 검사
for row_index, value in journal["department_code"].items():
    if pd.isna(value) or str(value) not in valid_department_codes:
        validation_errors.append({
            "row_number": row_index + 2,
            "voucher_id": journal.loc[row_index, "voucher_id"],
            "column": "department_code",
            "invalid_value": value,
            "error_type": "존재하지 않는 부서"
        })

# 오류 목록을 데이터프레임으로 변환
validation_result = pd.DataFrame(validation_errors)

print("발견된 마스터 데이터 오류 수:", len(validation_result))

# 오류가 있으면 상세 내용 출력
if len(validation_result) > 0:
    display(validation_result)
else:
    print("모든 계정과목·거래처·부서 코드가 기준표와 일치합니다.")

발견된 마스터 데이터 오류 수: 0
모든 계정과목·거래처·부서 코드가 기준표와 일치합니다.
